In [14]:
import torch
import math

In [8]:

a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
b = torch.randn(3, 5)
c = b.view(5, 3)
d = c.unsqueeze(0)


tensor([[[ 0.1597, -1.2537,  0.2613],
         [-0.1472,  0.8474, -0.7097],
         [-0.0424,  0.0964,  0.4202],
         [ 0.3734,  0.3396, -1.3973],
         [ 0.6803,  0.6671,  0.0455]]])

In [ ]:
a = torch.randn(5, 3)
a.reshape(3, 5)
a.unsqueeze(0)


2 урок

In [16]:

x = torch.tensor([2.0], requires_grad=True)
y = x * 3
loss = (y - 10)**2

loss.backward()
print(x.grad)       # градиент d(loss)/d(x)


tensor([-24.])


backward() . пайторч вычисляет градиенты для всех тензоров с requires_grad=True,
Результат накапливается в атрибуте .grad
у тензоров

In [18]:
x = torch.tensor([2.0], requires_grad=True)
y = x**3 + 2*x
y.backward()
print(x.grad)

tensor([14.])


x.grad содержит d(y)/d(x) — градиент целевой функции (обычно лосса) по каждому элементу тензора x.

такой же формы.

3 урок. linspace делает одинаковое расстояние

In [3]:
import torch
x = torch.linspace(-1, 1, 50)
y = 3*x + 1  + 0.1*torch.randn_like(x)   # шум добавлен
#y^​=w1​x1​+w2​x2​+...+w50​x50​+b могло быть! но это не тот случай
w = torch.rand(1, requires_grad=True)
b = torch.rand(1, requires_grad=True)
lr = 0.1
for _ in range(200):
    y_pred = w*x + b

    loss = (y_pred - y).pow(2).mean()
    loss.backward()
    
    with torch.no_grad():
        w -= lr*w.grad
        b -= lr*b.grad
    
    w.grad.zero_()
    b.grad.zero_() 

print(x, y)
print(w, b)

tensor([-1.0000, -0.9592, -0.9184, -0.8776, -0.8367, -0.7959, -0.7551, -0.7143,
        -0.6735, -0.6327, -0.5918, -0.5510, -0.5102, -0.4694, -0.4286, -0.3878,
        -0.3469, -0.3061, -0.2653, -0.2245, -0.1837, -0.1429, -0.1020, -0.0612,
        -0.0204,  0.0204,  0.0612,  0.1020,  0.1429,  0.1837,  0.2245,  0.2653,
         0.3061,  0.3469,  0.3878,  0.4286,  0.4694,  0.5102,  0.5510,  0.5918,
         0.6327,  0.6735,  0.7143,  0.7551,  0.7959,  0.8367,  0.8776,  0.9184,
         0.9592,  1.0000]) tensor([-1.9861, -1.8617, -1.7427, -1.6656, -1.4594, -1.5706, -1.2889, -1.0300,
        -0.9758, -1.0057, -0.8352, -0.6077, -0.5020, -0.4192, -0.3884, -0.0332,
        -0.0632,  0.0886,  0.2514,  0.4223,  0.4149,  0.6729,  0.7433,  0.7626,
         0.8395,  1.1597,  1.2247,  1.3827,  1.3314,  1.5991,  1.7176,  2.0124,
         1.8996,  2.0588,  2.1250,  2.1388,  2.3183,  2.5310,  2.6531,  2.6215,
         2.9271,  3.1737,  3.1872,  3.3561,  3.4270,  3.4670,  3.6370,  3.6024,
         3.85

with torch.no_grad() выключает построение вычислительного графа.

Глава 2. Модуль torch.nn

Основные компоненты:

nn.Module — базовый класс для моделей
nn.Linear — линейный слой
Переопределение forward
Параметры модели (model.parameters())
Лоссы из torch.nn
Оптимизаторы из torch.optim

2.1. Класс nn.Module

Это базовый класс, от которого наследуются все нейросети в PyTorch.

In [4]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # один вход → один выход

    def forward(self, x):
        return self.linear(x)

В __init__ объявляются слои.
В forward описывается, как данные проходят через модель.
Автоматически создаются параметры (weight, bias).
Autograd будет отслеживать параметры и вычислять градиенты.

In [6]:
model = MyModel()
x = torch.tensor([[0.5]])   # shape (batch, features)
y_pred = model(x)

print(y_pred)

tensor([[0.5096]], grad_fn=<AddmmBackward0>)


In [9]:
loss_fn = nn.MSELoss()
#лосс

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
#оптимизатор

In [10]:
for _ in range(200):
    y_pred = model(x)
    loss = loss_fn(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([50])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [13]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # один вход → один выход

    def forward(self, x):
        return self.linear(x)

x = torch.linspace(-1, 1, 50).unsqueeze(1)
y = 3*x + 1 + 0.1*torch.randn_like(x)

model = MyModel()

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)


for _ in range(200):
    y_pred = model(x)
    loss = loss_fn(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[2.9731]], requires_grad=True)
Parameter containing:
tensor([1.0151], requires_grad=True)
